In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Initialize Spark Session (Already available as 'spark' in Databricks notebooks)
spark = SparkSession.builder.appName("RideSharing_Medallion_Pipeline").getOrCreate()

# Task 1: Read all datasets using PySpark DataFrames from Databricks Catalog
# Note: Stripping the 1, 2, 3 list numbers from your prompt
drivers_df = spark.table("any.default.drivers")
trip_logs_df = spark.table("any.default.trip_logs")
trips_df = spark.table("any.default.trips")

# Task 2: Store raw data into Bronze layer without transformations
# Saving as Delta tables in Databricks
drivers_df.write.format("delta").mode("overwrite").saveAsTable("any.default.bronze_drivers")
trip_logs_df.write.format("delta").mode("overwrite").saveAsTable("any.default.bronze_trip_logs")
trips_df.write.format("delta").mode("overwrite").saveAsTable("any.default.bronze_trips")

In [0]:
# Read from Bronze layer
bronze_drivers = spark.table("any.default.bronze_drivers")
bronze_trip_logs = spark.table("any.default.bronze_trip_logs")
bronze_trips = spark.table("any.default.bronze_trips")

# Task 3 & 14: Perform joins using optimization techniques (Broadcast Join)
joined_df = bronze_trips \
    .join(F.broadcast(bronze_drivers), "driver_id", "left") \
    .join(bronze_trip_logs, "trip_id", "left")

# Task 4 & 6: Clean data, handle nulls, and filter invalid data
cleaned_df = joined_df \
    .filter(F.col("distance_km") >= 0) \
    .filter((F.col("trip_status") == "Cancelled") | (F.col("start_time").isNotNull() & F.col("end_time").isNotNull())) \
    .fillna({"delay_minutes": 0, "fare_amount": 0.0})

# Task 5: Create derived columns (trip duration and completion flag)
silver_df = cleaned_df \
    .withColumn("is_completed", F.when(F.col("trip_status") == "Completed", 1).otherwise(0)) \
    .withColumn("trip_duration_mins", 
                F.round((F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 60, 2))

# Task 7: Write cleaned data into Silver layer (Using Delta instead of Parquet for Databricks optimization)
silver_df.write.format("delta").mode("overwrite").saveAsTable("any.default.silver_enriched_trips")

In [0]:
# Read from Silver Layer
silver_data = spark.table("any.default.silver_enriched_trips")

# Task 8, 9, 11: Driver performance, cancellation rates, and revenue
driver_metrics_df = silver_data.groupBy("driver_id", "name", "city", "rating").agg(
    F.count("trip_id").alias("total_trips"),
    F.sum("is_completed").alias("completed_trips"),
    F.sum("cancellation_flag").alias("cancelled_trips"),
    F.round((F.sum("cancellation_flag") / F.count("trip_id")) * 100, 2).alias("cancellation_rate_pct"),
    F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
    F.round(F.avg("delay_minutes"), 2).alias("avg_delay_mins")
)

# Task 15: Use window functions to rank drivers based on performance (Revenue)
window_spec = Window.partitionBy("city").orderBy(F.col("total_revenue").desc())

driver_performance_gold = driver_metrics_df \
    .withColumn("city_rank_by_revenue", F.rank().over(window_spec))

# Task 10: Identify high-demand pickup locations
high_demand_locations_gold = silver_data.groupBy("pickup_location").agg(
    F.count("trip_id").alias("total_requests"),
    F.sum("is_completed").alias("successful_pickups"),
    F.round(F.sum("fare_amount"), 2).alias("revenue_generated")
).orderBy(F.col("total_requests").desc())

# Task 12: Store aggregated results in Gold layer
driver_performance_gold.write.format("delta").mode("overwrite").saveAsTable("any.default.gold_driver_performance")
high_demand_locations_gold.write.format("delta").mode("overwrite").saveAsTable("any.default.gold_high_demand_locations")

In [0]:
# Task 13: Validate data consistency across layers
bronze_count = spark.table("any.default.bronze_trips").count()
silver_count = spark.table("any.default.silver_enriched_trips").count()

print(f"Total Raw Trips (Bronze): {bronze_count}")
print(f"Total Valid Trips (Silver): {silver_count}")

# Basic assertion to ensure data integrity
if silver_count <= bronze_count:
    print("Data validation passed successfully. Silver records do not exceed Bronze records.")
else:
    print("WARNING: Silver layer has more records than Bronze. Check joins for potential duplicates!")

Total Raw Trips (Bronze): 150
Total Valid Trips (Silver): 150
Data validation passed successfully. Silver records do not exceed Bronze records.
